Imports Python libraries

In [1]:
import pandas as pd


Loads data from wildfire dataset

In [2]:
df = pd.read_csv("../data/wildfire_data.csv")
df.head()

/var/folders/bk/jmg3rr0j58s3yrt8fdrdc1ph0000gn/T/ipykernel_25953/3130476503.py:1: DtypeWarning: Columns (0: INC_NUM, 1: COMPLEX_NA, 2: COMPLEX_ID) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/wildfire_data.csv")


,OBJECTID,Year,AGENCY,UNIT_ID,FIRE_NAME,INC_NUM,ALARM_DATE,CONT_DATE,CAUSE,C_METHOD,...,longitude,zip,Alarm_Date2,year_month,AGENCY_ID,FIRE_NAME_ID,avg_tmax_c,avg_tmin_c,tot_prcp_mm,station
0,1653.0,2019.0,CDF,MEU,GRADE,8307.0,7/16/17,7/21/17,10.0,7.0,...,-123.266813,95470.0,2017-07-16,2017-07,0.0,92.0,NaN,NaN,NaN,NaN
1,1982.0,2018.0,CCO,VNC,SOUTH,113365.0,12/21/17,12/21/17,14.0,6.0,...,-119.051707,93060.0,2017-12-21,2017-12,2.0,102.0,NaN,NaN,NaN,NaN
2,1980.0,2018.0,CCO,VNC,BALCOM,113691.0,12/22/17,12/22/17,14.0,6.0,...,-118.977639,93066.0,2017-12-22,2017-12,2.0,1400.0,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,90001.0,NaN,2018-01,NaN,NaN,21.748387,11.432258,35.5,72295.0
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,90002.0,NaN,2018-01,NaN,NaN,21.748387,11.432258,35.5,72295.0


Prints size of dataset

In [3]:
df.shape

(125476, 30)

Prints names of columns

In [4]:
df.columns

Index(['OBJECTID', 'Year', 'AGENCY', 'UNIT_ID', 'FIRE_NAME', 'INC_NUM',
       'ALARM_DATE', 'CONT_DATE', 'CAUSE', 'C_METHOD', 'OBJECTIVE',
       'GIS_ACRES', 'COMMENTS', 'COMPLEX_NA', 'FIRE_NUM', 'COMPLEX_ID',
       'DECADES', 'Shape__Are', 'Shape__Len', 'latitude', 'longitude', 'zip',
       'Alarm_Date2', 'year_month', 'AGENCY_ID', 'FIRE_NAME_ID', 'avg_tmax_c',
       'avg_tmin_c', 'tot_prcp_mm', 'station'],
      dtype='str')

This restricts the data to the years to be modeled off of

In [5]:
df['Year'] = df['Year'].ffill()
df['Year'] = df['Year'].astype(int)

Takes 2599 unique zip codes and turns it into 66 regions (easier to compute and predict given present ability of QML)

In [6]:
df['zip'] = df['zip'].astype(str).str.zfill(5)
df['zip3'] = df['zip'].str[:3]
df['zip3'].nunique()

66

Converts the 66 geographic regions into 65 columns with binary values

In [7]:
df = pd.get_dummies(df, columns=['zip3'], drop_first=True)
df = df.drop(columns=['zip'])

Creates the target variable "high risk" to define what this model predicts (predicts when there is a high risk fire); this is a new column in the dataset and classifies all answers as binary

In [8]:
df['high_risk'] = (df['GIS_ACRES']>1000).astype(int)

Tells how many missing values in each column

In [9]:
df.isnull().sum()

OBJECTID     123258
Year              0
AGENCY       123258
UNIT_ID      123267
FIRE_NAME    123265
              ...  
zip3_960          0
zip3_961          0
zip3_975          0
zip3_976          0
high_risk         0
Length: 95, dtype: int64

Defines features to make prediction

In [10]:
base_features = ['avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm']
zip3_cols = [col for col in df.columns if col.startswith('zip3_')]
features = base_features + zip3_cols
target = 'high_risk'

Creates df_model with only necessary columns

In [11]:
df_model = df[features + [target]+ ['Year']].copy()
df_model.shape

(125476, 70)

Trains the ML model on the data from 2018-2022 and tests its accuracy on 2023 data
X_train: 2018-2022 features
X_test: 2023 features
y_train: 2018-2022 labels
y_test: 2023 labels

In [12]:
train_df = df_model[(df_model['Year'] >= 2018) & (df_model['Year'] <= 2022)]
test_df = df_model[df_model['Year'] == 2023]
X_train = train_df[features]
X_test = test_df[features]
y_train = train_df[target]
y_test = test_df[target]

In [13]:
X_train.isna().sum()

avg_tmax_c     588
avg_tmin_c     588
tot_prcp_mm    546
zip3_890         0
zip3_894         0
              ... 
zip3_959         0
zip3_960         0
zip3_961         0
zip3_975         0
zip3_976         0
Length: 68, dtype: int64

When data is missing (~550 for temps and precipitation), does risk behave differently?

In [14]:
for col in ['avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm']:
    df_model[col + '_missing'] = df_model[col].isna().astype(int)

Fills in missing weather info with medians (standard baseline robust to outliers)
Helpful to know geography and time and not be restricted by some missing weather vals

In [15]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)

Scales so precipitation and temperature are both adequately considered

In [16]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

Imports models + metrics

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

Weighted Logistic Regression

In [19]:
log_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)

log_pred = log_model.predict(X_test_scaled)
log_proba = log_model.predict_proba(X_test_scaled)[:, 1]

print("Weighted Logistic Regression Results")
print("Accuracy: ", accuracy_score(y_test, log_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, log_pred))
print("Classification Report:")
print(classification_report(y_test, log_pred))

Weighted Logistic Regression Results
Accuracy:  0.3415492957746479
Confusion Matrix:
[[ 71 183]
 [  4  26]]
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.28      0.43       254
           1       0.12      0.87      0.22        30

    accuracy                           0.34       284
   macro avg       0.54      0.57      0.32       284
weighted avg       0.86      0.34      0.41       284

